In [16]:
import importlib

import carro_rl
import carro_rl.entorno

importlib.reload(carro_rl.entorno)


<module 'carro_rl.entorno' from 'C:\\Users\\Usuario\\Documents\\Universidad UPB\\Sexto-septimo Semestre\\Técnicas avanzadas IA\\proyecto_carro_reinforcement_learning\\carro_rl\\entorno.py'>

In [17]:
import gymnasium as gym
import numpy as np
import matplotlib.pyplot as plt

from stable_baselines3 import DQN, SAC
from stable_baselines3.common.monitor import Monitor
from stable_baselines3.common.evaluation import evaluate_policy

In [19]:
import importlib

import carro_rl.entorno

importlib.reload(carro_rl.entorno)

from pathlib import Path

pista = "ovalo"
pasos_entrenamiento = 200_000

# 1. Crear el entorno de entrenamiento
env = Monitor(gym.make(
    "CarroPista-v0",
    pista=pista,
    peso_velocidad=2.0,
    penal_choque=10.0,
))

# 2. Crear un modelo SAC nuevo. No continua sac_carro_v3.
modelo = SAC(
    "MlpPolicy",
    env,
    gamma=0.995,
    learning_rate=3e-4,
    buffer_size=200_000,
    learning_starts=20_000,
    batch_size=256,
    tau=0.005,
    ent_coef=0.05,
    train_freq=1,
    gradient_steps=1,
    policy_kwargs=dict(net_arch=[256, 256]),
    verbose=1,
    seed=0,
    device="cpu",
)

# 3. Entrenar
modelo.learn(
    total_timesteps=pasos_entrenamiento,
    reset_num_timesteps=True,
)

# 4. Guardar el modelo entrenado
raiz = Path.cwd()

if raiz.name == "Algoritmo_solucion":
    raiz = raiz.parent

ruta_modelo = raiz / "Algoritmo_solucion" / "sac_carro_v4"
ruta_modelo.parent.mkdir(parents=True, exist_ok=True)

modelo.save(str(ruta_modelo))
print(f"Modelo guardado en: {ruta_modelo}.zip")

# 5. Evaluar en cinco episodios independientes
eval_env = Monitor(gym.make(
    "CarroPista-v0",
    pista=pista,
    peso_velocidad=2.0,
    penal_choque=10.0,
))

media, desv = evaluate_policy(
    modelo,
    eval_env,
    n_eval_episodes=5,
    deterministic=True,
)

print(f"Evaluación: retorno {media:.2f} ± {desv:.2f}")

# 6. Analizar un episodio y comprobar el resultado
obs, info = eval_env.reset(seed=0)
total = 0.0
fin = False
term = False
trunc = False

while not fin:
    accion, _ = modelo.predict(obs, deterministic=True)
    obs, recompensa, term, trunc, info = eval_env.step(accion)
    total += recompensa
    fin = term or trunc

print(
    f"pista={info['pista']} "
    f"vuelta={info['vuelta_completada']} "
    f"choque={info['choque']} "
    f"progreso={info['progreso']:.2f} "
    f"t={info['tiempo']:.1f}s "
    f"retorno={total:.2f} "
    f"truncado={trunc}"
)

eval_env.close()
env.close()


Using cpu device
Wrapping the env in a DummyVecEnv.
---------------------------------
| rollout/           |          |
|    ep_len_mean     | 985      |
|    ep_rew_mean     | -10.7    |
| time/              |          |
|    episodes        | 4        |
|    fps             | 1324     |
|    time_elapsed    | 2        |
|    total_timesteps | 3940     |
---------------------------------
---------------------------------
| rollout/           |          |
|    ep_len_mean     | 964      |
|    ep_rew_mean     | -10.6    |
| time/              |          |
|    episodes        | 8        |
|    fps             | 1333     |
|    time_elapsed    | 5        |
|    total_timesteps | 7713     |
---------------------------------
---------------------------------
| rollout/           |          |
|    ep_len_mean     | 1.09e+03 |
|    ep_rew_mean     | -10.7    |
| time/              |          |
|    episodes        | 12       |
|    fps             | 1331     |
|    time_elapsed    | 9      

In [9]:
print("Espacio de acciones:", env.action_space)
print("Espacio de observaciones:", env.observation_space)


Espacio de acciones: Box(-1.0, 1.0, (2,), float32)
Espacio de observaciones: Box([ 0.  0.  0.  0.  0.  0.  0. -1. -1. -1.], 1.0, (10,), float32)
